# AnyUp3D — Checkpoint Comparison Notebook

Scans a directory of `.pth` checkpoints, runs every one on a test video set,
computes all project metrics, and produces ranked plots + a summary table.

**Metrics computed per checkpoint:**
| Metric | What it measures |
|---|---|
| `cos_sim` | Mean cosine similarity to frozen VideoMAE teacher features (↑ better) |
| `cos_mse` | Reconstruction loss (↓ better) |
| `temporal_var` | Feature variance across frames — lower = more stable (↓ better) |
| `cosine_drift` | Frame-to-frame feature drift (↓ better) |
| `input_consistency` | How well high-res output downsamples back to input (↓ better) |

Additionally, any **logged scalars** stored inside the checkpoint dict
(e.g. loss curves from training) are extracted and plotted automatically.


## 1 · Configuration
Edit the paths and knobs below — everything else runs automatically.

In [ ]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR   = "/content/drive/MyDrive/anyup3d/checkpoints/run_01"  # folder with .pth files
VIDEO_DIR        = "/content/test_videos"   # folder with .mp4 / .avi test clips
                                            # OR set VIDEO_DIR = None to use synthetic stubs

# ── EVAL KNOBS ────────────────────────────────────────────────────────────────
NUM_TEST_CLIPS   = 8      # ↓ reduce to 2-3 for a quick sanity check
NUM_FRAMES       = 8      # ↓ must match what your model was trained on; reduce if OOM
                           #   also controls temporal metric granularity
SPATIAL_SIZE     = 224    # ↓ H=W for resize; reduce to 112 if OOM
BATCH_SIZE       = 1      # ↓ keep at 1 on Colab T4; increase only on A100
ENCODER_MODEL    = "MCG-NJU/videomae-base"  # frozen teacher — same as training
DEVICE           = "cuda"  # "cpu" if no GPU

# ── VIS KNOBS ─────────────────────────────────────────────────────────────────
TOP_K_CKPTS_PCA  = 3      # show PCA feature maps for the top-K and bottom-1 checkpoints
STATIC_THRESHOLD = 0.02   # RGB-diff gate for temporal variance (same as training default)

print("Config ready.")


## 2 · Installs & Imports

In [ ]:
# Install AnyUp from Hub (skip if already installed in your Colab)
import subprocess, sys
try:
    import anyup
    print("anyup already installed")
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    "git+https://github.com/wimmerth/anyup.git", "-q"], check=True)
    print("anyup installed")

# Standard libs
import os, glob, json, time, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.decomposition import PCA  # for feature visualisation

import torch
import torch.nn.functional as F
from torch import Tensor
from torchvision import transforms

warnings.filterwarnings("ignore")
device = torch.device(DEVICE if torch.cuda.is_available() else "cpu")
print(f"Running on: {device}")


## 3 · Discover Checkpoints

In [ ]:
ckpt_paths = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, "**/*.pth"), recursive=True)
                  + glob.glob(os.path.join(CHECKPOINT_DIR, "*.pth")))

if not ckpt_paths:
    raise FileNotFoundError(f"No .pth files found in {CHECKPOINT_DIR}")

print(f"Found {len(ckpt_paths)} checkpoints:")
for p in ckpt_paths:
    size_mb = os.path.getsize(p) / 1e6
    print(f"  {Path(p).name:50s}  {size_mb:.1f} MB")


## 4 · Load Frozen Teacher (VideoMAE)

In [ ]:
from transformers import VideoMAEModel, VideoMAEFeatureExtractor

print(f"Loading teacher: {ENCODER_MODEL} ...")
teacher_extractor = VideoMAEFeatureExtractor.from_pretrained(ENCODER_MODEL)
teacher_model     = VideoMAEModel.from_pretrained(ENCODER_MODEL).to(device)
teacher_model.eval()
for p in teacher_model.parameters():
    p.requires_grad_(False)

# VideoMAE patch grid: 16 frames → (8, 14, 14) tokens for base model
# ↓ T_teacher must be exactly 16 for VideoMAE; we tile our frames if T < 16
T_TEACHER = 16          # fixed by VideoMAE architecture — do NOT change
TOK_T     = T_TEACHER // 2   # temporal token stride (tubelet=2) → 8 token frames
TOK_H     = SPATIAL_SIZE // 16  # ↓ spatial token stride 16 — depends on SPATIAL_SIZE
TOK_W     = SPATIAL_SIZE // 16
ENC_DIM   = 768         # VideoMAE-base hidden dim — used to size the feature volumes

print(f"Teacher ready. Token grid: ({TOK_T}, {TOK_H}, {TOK_W}), dim={ENC_DIM}")


## 5 · Build Test Dataset

In [ ]:
# ── normalisation (ImageNet stats, same as training) ───────────────────────────
normalize = transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])

def load_clip_from_file(video_path: str, num_frames: int, spatial_size: int) -> Tensor:
    """
    Read `num_frames` evenly-spaced frames from a video file.
    Returns (1, num_frames, H, W, 3) float32 in [0,1].

    Requires torchvision >= 0.13 with video_reader backend, OR falls back
    to imageio / decord if available.
    """
    try:
        import torchvision.io as tio
        vframes, _, _ = tio.read_video(video_path, pts_unit="sec")  # (T, H, W, C) uint8
        indices = torch.linspace(0, len(vframes)-1, num_frames).long()  # ↓ evenly spaced
        frames  = vframes[indices].float() / 255.0                       # (T, H, W, 3)
    except Exception:
        # decord fallback
        import decord
        decord.bridge.set_bridge("torch")
        vr = decord.VideoReader(video_path)
        indices = torch.linspace(0, len(vr)-1, num_frames).long().tolist()
        frames  = vr.get_batch(indices).float() / 255.0    # (T, H, W, 3)

    # Resize spatial dims
    frames = frames.permute(0,3,1,2)                                     # (T, 3, H, W)
    frames = F.interpolate(frames, size=(spatial_size, spatial_size),
                           mode="bilinear", align_corners=False)          # ↓ spatial_size
    frames = frames.permute(0,2,3,1)                                     # (T, H, W, 3)
    return frames.unsqueeze(0)                                            # (1, T, H, W, 3)


def make_stub_clip(num_frames: int, spatial_size: int) -> Tensor:
    """Synthetic random clip for smoke-testing without real video files."""
    return torch.rand(1, num_frames, spatial_size, spatial_size, 3)      # (1, T, H, W, 3)


# ── Collect test clips ────────────────────────────────────────────────────────
test_clips = []  # list of (1, T, H, W, 3) tensors

if VIDEO_DIR is not None and os.path.isdir(VIDEO_DIR):
    video_files = sorted(
        glob.glob(os.path.join(VIDEO_DIR, "*.mp4")) +
        glob.glob(os.path.join(VIDEO_DIR, "*.avi")) +
        glob.glob(os.path.join(VIDEO_DIR, "*.mov"))
    )[:NUM_TEST_CLIPS]  # ↓ cap at NUM_TEST_CLIPS

    for vf in video_files:
        try:
            clip = load_clip_from_file(vf, NUM_FRAMES, SPATIAL_SIZE)
            test_clips.append(clip)
            print(f"  Loaded: {Path(vf).name}  shape={list(clip.shape)}")
        except Exception as e:
            print(f"  SKIP {Path(vf).name}: {e}")

if len(test_clips) < NUM_TEST_CLIPS:
    n_stubs = NUM_TEST_CLIPS - len(test_clips)
    print(f"Padding with {n_stubs} synthetic stub clip(s) (VIDEO_DIR empty or not enough files).")
    for _ in range(n_stubs):
        test_clips.append(make_stub_clip(NUM_FRAMES, SPATIAL_SIZE))

print(f"\nTest set: {len(test_clips)} clips, each {list(test_clips[0].shape)}")


## 6 · Helper Functions

In [ ]:
# ─── Teacher feature extraction ───────────────────────────────────────────────

@torch.no_grad()
def extract_teacher_features(frames: Tensor) -> Tensor:
    """
    frames: (B, T, H, W, 3) float32 [0,1]
    Returns (B, T_tok, H_tok, W_tok, ENC_DIM) — spatiotemporal token volume.

    VideoMAE requires exactly 16 frames. We tile if T < 16.
    """
    B, T, H, W, C = frames.shape
    # Normalise and rearrange to (B, T, C, H, W)
    f = frames.permute(0,1,4,2,3)              # (B, T, 3, H, W)
    f = normalize(f.reshape(B*T, 3, H, W))     # normalise each frame independently
    f = f.reshape(B, T, 3, H, W)

    # Tile to reach T_TEACHER=16 frames if needed
    if T < T_TEACHER:                          # ↓ T_TEACHER fixed at 16 by VideoMAE
        repeats = (T_TEACHER + T - 1) // T
        f = f.repeat(1, repeats, 1, 1, 1)[:, :T_TEACHER]  # (B, 16, 3, H, W)

    # VideoMAE expects (B, C, T, H, W) as pixel_values
    pv = f.permute(0, 2, 1, 3, 4).to(device)   # (B, 3, 16, H, W)
    out = teacher_model(pixel_values=pv)        # last_hidden_state: (B, N_tok, ENC_DIM)

    # Reshape tokens to spatial volume
    N = TOK_T * TOK_H * TOK_W                  # total tokens
    feats = out.last_hidden_state[:, :N, :]    # (B, N, ENC_DIM) — trim cls token if present
    feats = feats.reshape(B, TOK_T, TOK_H, TOK_W, ENC_DIM)  # (B, T_tok, H_tok, W_tok, C)

    # Trim temporal dim back to T tokens (aligns with our num_frames)
    T_tok_out = min(TOK_T, T)                  # ↓ use original T frames worth of tokens
    feats = feats[:, :T_tok_out]               # (B, T_tok_out, H_tok, W_tok, C)
    return feats


# ─── Metric helpers ───────────────────────────────────────────────────────────

def cosine_similarity_to_target(pred: Tensor, target: Tensor) -> float:
    """Mean cosine similarity between pred and target over all spatiotemporal positions."""
    pred_f   = pred.reshape(-1, pred.shape[-1])     # (N, C)
    target_f = target.reshape(-1, target.shape[-1]) # (N, C)
    return F.cosine_similarity(pred_f, target_f, dim=-1).mean().item()  # scalar ↑ better


def cos_mse(pred: Tensor, target: Tensor) -> float:
    """Reconstruction loss (lower = better)."""
    pred_f   = pred.reshape(-1, pred.shape[-1])
    target_f = target.reshape(-1, target.shape[-1])
    cos_loss = (1.0 - F.cosine_similarity(pred_f, target_f, dim=-1)).mean()
    mse_loss = F.mse_loss(pred_f, target_f)
    return (cos_loss + mse_loss).item()              # scalar ↓ better


def temporal_feature_variance(feats: Tensor, frames: Tensor,
                               static_threshold: float = 0.02) -> float:
    """
    Mean per-position variance of features across T frames.
    Optionally gates on static scene regions (low RGB diff between frames).
    Returns scalar — lower = more temporally stable.
    """
    B, T, H, W, C = feats.shape
    if T < 2:
        return 0.0

    # Variance across T dim → (B, H, W, C) then mean over C, H, W
    var = feats.var(dim=1)                           # (B, H, W, C)

    if frames is not None:
        # Static gate: mean RGB diff across adjacent frame pairs per spatial position
        # ↓ frames: (B, T, H, W, 3) — resize to feature resolution for gate
        frames_resized = F.interpolate(
            frames.permute(0,1,4,2,3).reshape(B*T, 3, frames.shape[2], frames.shape[3]),
            size=(H, W), mode="bilinear", align_corners=False,
        ).reshape(B, T, 3, H, W).permute(0,1,3,4,2)  # (B, T, H, W, 3)

        rgb_diff = (frames_resized[:, 1:] - frames_resized[:, :-1]).abs().mean(dim=(1,4))  # (B, H, W)
        static_mask = (rgb_diff.mean(dim=0) < static_threshold).float()                    # (H, W)

        # Weight variance by static mask
        masked_var = var.mean(dim=-1) * static_mask.unsqueeze(0)   # (B, H, W)
        n_static   = static_mask.sum().clamp(min=1.0)
        return (masked_var.sum() / (B * n_static)).item()

    return var.mean().item()                         # fallback: unmasked


def temporal_cosine_drift(feats: Tensor) -> float:
    """Mean 1-cosine_sim between adjacent frames. Lower = more coherent."""
    B, T, H, W, C = feats.shape
    if T < 2:
        return 0.0
    f_curr = feats[:, 1:].reshape(-1, C)             # (B*(T-1)*H*W, C)
    f_prev = feats[:, :-1].reshape(-1, C)
    drift  = 1.0 - F.cosine_similarity(f_curr, f_prev, dim=-1)
    return drift.mean().item()                        # ↓ better


def input_consistency(pred: Tensor, p_lores: Tensor) -> float:
    """
    Downsample pred to p_lores resolution and compare.
    pred:    (B, T, H, W, C)  high-res
    p_lores: (B, T, h, w, C)  teacher features at token resolution
    """
    B, T, H, W, C = pred.shape
    _, _, h, w, _ = p_lores.shape
    # Spatial pool per frame
    pred_rs    = pred.permute(0,1,4,2,3).reshape(B*T, C, H, W)       # (B*T, C, H, W)
    pred_down  = F.adaptive_avg_pool2d(pred_rs, (h, w))               # (B*T, C, h, w)
    pred_down  = pred_down.reshape(B, T, C, h, w).permute(0,1,3,4,2) # (B, T, h, w, C)
    return cos_mse(pred_down, p_lores)                                 # ↓ better


print("Metric helpers ready.")


In [ ]:
# ─── Load AnyUp3D from a checkpoint ──────────────────────────────────────────

def load_anyup3d(ckpt_path: str):
    """
    Loads AnyUp (3D-capable) from a .pth checkpoint.
    Handles both raw state-dict files and full checkpoint dicts
    (i.e. those saved with optimizer state, step, etc.).

    Returns (model, metadata_dict).
    """
    raw = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # Detect checkpoint format
    if isinstance(raw, dict) and "anyup" in raw:
        state_dict = raw["anyup"]                       # training checkpoint format
        meta = {k: v for k, v in raw.items() if k != "anyup"}
    elif isinstance(raw, dict) and "state_dict" in raw:
        state_dict = raw["state_dict"]
        meta = {k: v for k, v in raw.items() if k != "state_dict"}
    elif isinstance(raw, dict) and all(isinstance(v, Tensor) for v in raw.values()):
        state_dict = raw                                # bare state-dict
        meta = {}
    else:
        # Might be the full dict itself — try treating as state_dict
        state_dict = raw
        meta = {}

    # Build model via PyTorch Hub (same as training pipeline)
    model = torch.hub.load("wimmerth/anyup", "anyup", pretrained=False, verbose=False)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    if missing:
        print(f"    Missing keys  : {len(missing)}")
    if unexpected:
        print(f"    Unexpected keys: {len(unexpected)}")

    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)

    return model, meta


# ─── Run model on a single clip ───────────────────────────────────────────────

@torch.no_grad()
def run_model(model, frames: Tensor) -> Tensor:
    """
    frames: (1, T, H, W, 3) float32 [0,1]
    Returns upsampled features — shape depends on model; we expect (1, T, H', W', C).

    AnyUp forward: needs (B, C, H, W) per-frame inputs.
    We call it frame-by-frame and stack back to (B, T, H', W', C).
    """
    B, T, H, W, _ = frames.shape      # ↑ T set by NUM_FRAMES
    out_frames = []

    # Normalise frames → (B*T, 3, H, W)
    f = frames.permute(0,1,4,2,3).reshape(B*T, 3, H, W).to(device)  # ↑ 3=RGB channels
    f = normalize(f)

    # Run model per-frame (AnyUp2D-style) — replace with AnyUp3D joint call if available
    for t in range(T):                 # ↓ T loop — most compute is here
        frame_t = f[t:t+1]            # (1, 3, H, W)
        pred_t  = model(frame_t)       # (1, C, H', W')  ← AnyUp output
        out_frames.append(pred_t.permute(0,2,3,1))   # (1, H', W', C)

    # Stack to (B, T, H', W', C)
    pred = torch.stack(out_frames, dim=1)             # (1, T, H', W', C)
    return pred


print("Model loader ready.")


## 7 · Evaluate All Checkpoints

This is the main loop — loads each checkpoint, runs it on every test clip, and accumulates metrics.

In [ ]:
results = []   # list of dicts, one per checkpoint

for ckpt_path in ckpt_paths:
    ckpt_name = Path(ckpt_path).stem
    print(f"\n{'='*60}")
    print(f"  Checkpoint: {ckpt_name}")
    print(f"{'='*60}")

    # ── Load model ────────────────────────────────────────────────────────────
    t0 = time.time()
    try:
        model, meta = load_anyup3d(ckpt_path)
    except Exception as e:
        print(f"  ERROR loading: {e}")
        continue

    # Extract training metadata from checkpoint
    global_step   = meta.get("global_step", None)
    t_stage       = meta.get("t_stage", None)         # curriculum T value at save time
    train_losses  = meta.get("loss_history", {})       # dict of lists if logged

    print(f"  global_step={global_step}  t_stage={t_stage}  load_time={time.time()-t0:.1f}s")

    # ── Run on test clips ─────────────────────────────────────────────────────
    clip_metrics = defaultdict(list)

    for i, clip in enumerate(test_clips):
        clip_dev = clip.to(device)                     # (1, T, H, W, 3)

        # Teacher features (ground-truth)
        try:
            gt_feats = extract_teacher_features(clip_dev)   # (1, T_tok, H_tok, W_tok, C)
        except Exception as e:
            print(f"    clip {i}: teacher failed — {e}")
            continue

        # Model forward
        try:
            pred = run_model(model, clip_dev)               # (1, T, H', W', C)
        except Exception as e:
            print(f"    clip {i}: model forward failed — {e}")
            continue

        # Align spatial resolution between pred and gt_feats for comparison
        B, T_pred, H_pred, W_pred, C_pred = pred.shape
        _, T_gt,   H_gt,   W_gt,   C_gt   = gt_feats.shape

        # ↓ resize pred spatially to GT token resolution for metric computation
        T_min = min(T_pred, T_gt)
        pred_rs   = pred[:, :T_min].permute(0,1,4,2,3).reshape(-1, C_pred, H_pred, W_pred)
        pred_rs   = F.interpolate(pred_rs, size=(H_gt, W_gt),
                                  mode="bilinear", align_corners=False)
        pred_rs   = pred_rs.reshape(B, T_min, C_pred, H_gt, W_gt).permute(0,1,3,4,2)  # (B,T,h,w,C)
        gt_rs     = gt_feats[:, :T_min]                    # (B, T_min, H_gt, W_gt, C_gt)

        # ↓ project to same C if dims differ (shouldn't normally happen)
        if C_pred != C_gt:
            # mean-pool channels to smaller dim as a fallback
            C_min = min(C_pred, C_gt)
            pred_rs = pred_rs[..., :C_min]
            gt_rs   = gt_rs[...,   :C_min]

        # Compute all metrics
        clip_metrics["cos_sim"].append(cosine_similarity_to_target(pred_rs, gt_rs))
        clip_metrics["cos_mse"].append(cos_mse(pred_rs, gt_rs))
        clip_metrics["temporal_var"].append(
            temporal_feature_variance(pred_rs, clip_dev[:, :T_min], STATIC_THRESHOLD))
        clip_metrics["cosine_drift"].append(temporal_cosine_drift(pred_rs))
        clip_metrics["input_consistency"].append(input_consistency(pred_rs, gt_rs))

        print(f"    clip {i:02d}: cos_sim={clip_metrics['cos_sim'][-1]:.4f} "
              f"cos_mse={clip_metrics['cos_mse'][-1]:.4f} "
              f"temp_var={clip_metrics['temporal_var'][-1]:.5f} "
              f"drift={clip_metrics['cosine_drift'][-1]:.4f}")

    # Aggregate across clips (mean)
    if not clip_metrics["cos_sim"]:
        print("  No valid clips — skipping.")
        continue

    row = {
        "name":        ckpt_name,
        "path":        ckpt_path,
        "global_step": global_step,
        "t_stage":     t_stage,
        # ↑ higher is better
        "cos_sim":          np.mean(clip_metrics["cos_sim"]),
        # ↓ lower is better for all the rest
        "cos_mse":          np.mean(clip_metrics["cos_mse"]),
        "temporal_var":     np.mean(clip_metrics["temporal_var"]),
        "cosine_drift":     np.mean(clip_metrics["cosine_drift"]),
        "input_consistency":np.mean(clip_metrics["input_consistency"]),
        # raw for box plots
        "_raw": dict(clip_metrics),
        # training loss history if saved in ckpt
        "_train_losses": train_losses,
    }
    results.append(row)

    # Free GPU memory before next checkpoint
    del model
    torch.cuda.empty_cache()               # ↓ essential on Colab T4 between checkpoints

print(f"\n✓ Evaluated {len(results)} / {len(ckpt_paths)} checkpoints.")


## 8 · Summary Table

In [ ]:
# Build a clean DataFrame
df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                   for r in results])

# Normalise each metric to [0,1] and compute a composite score
# cos_sim: higher is better → invert for score computation
# the rest: lower is better → invert
if len(df) > 1:
    def norm(s, invert=False):
        mn, mx = s.min(), s.max()
        if mx == mn: return pd.Series([0.5]*len(s), index=s.index)
        n = (s - mn) / (mx - mn)
        return (1 - n) if invert else n

    df["score"] = (
        norm(df["cos_sim"])               * 0.35   # highest weight — primary reconstruction
      + norm(df["cos_mse"],    invert=True) * 0.25
      + norm(df["temporal_var"],invert=True)* 0.15
      + norm(df["cosine_drift"],invert=True)* 0.15
      + norm(df["input_consistency"],invert=True)*0.10
    )
    df = df.sort_values("score", ascending=False).reset_index(drop=True)

display_cols = ["name","global_step","t_stage","cos_sim","cos_mse",
                "temporal_var","cosine_drift","input_consistency"] + (
               ["score"] if "score" in df.columns else [])

styled = df[display_cols].style\
    .background_gradient(subset=["cos_sim"], cmap="Greens")\
    .background_gradient(subset=["cos_mse","temporal_var","cosine_drift","input_consistency"], cmap="RdYlGn_r")\
    .format({c: "{:.5f}" for c in ["cos_sim","cos_mse","temporal_var","cosine_drift","input_consistency"]}
            | ({"score":"{:.3f}"} if "score" in df.columns else {}))\
    .set_caption("Checkpoint Comparison — sorted by composite score (↑ better)")

display(styled)

# Print winner
if len(df) > 0:
    best = df.iloc[0]
    print(f"\n🏆 Best checkpoint: {best['name']}")
    if best["global_step"] is not None:
        print(f"   Step: {best['global_step']}  |  T-stage: {best['t_stage']}")
    print(f"   cos_sim={best['cos_sim']:.4f}  cos_mse={best['cos_mse']:.4f}  "
          f"temp_var={best['temporal_var']:.5f}  drift={best['cosine_drift']:.4f}")


## 9 · Metric Bar Charts

In [ ]:
metric_cfg = [
    ("cos_sim",           "Cosine Similarity to Teacher",    "↑ higher is better", True),
    ("cos_mse",           "Reconstruction Loss (cos+mse)",   "↓ lower is better",  False),
    ("temporal_var",      "Temporal Feature Variance",       "↓ lower is better",  False),
    ("cosine_drift",      "Temporal Cosine Drift",           "↓ lower is better",  False),
    ("input_consistency", "Input Consistency Loss",          "↓ lower is better",  False),
]

n_metrics = len(metric_cfg)
fig, axes = plt.subplots(1, n_metrics, figsize=(5*n_metrics, 5))
fig.suptitle("AnyUp3D Checkpoint Comparison", fontsize=14, fontweight="bold", y=1.02)

for ax, (col, title, direction, higher_better) in zip(axes, metric_cfg):
    vals   = df[col].values
    names  = [n[:25] for n in df["name"].values]    # truncate long names
    colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(vals)))
    if not higher_better:
        colors = colors[::-1]                        # green = low for "lower is better" metrics

    bars = ax.barh(range(len(names)), vals, color=colors, edgecolor="white", linewidth=0.5)

    # Annotate bars with value
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                f"{v:.4f}", va="center", ha="left", fontsize=7)

    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names, fontsize=8)
    ax.set_title(f"{title}\n{direction}", fontsize=9, pad=4)
    ax.invert_yaxis()                                # best at top (sorted by score)
    ax.spines[["top","right"]].set_visible(False)
    ax.tick_params(axis="x", labelsize=7)

plt.tight_layout()
plt.savefig("/tmp/checkpoint_bar_charts.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /tmp/checkpoint_bar_charts.png")


## 10 · Metric vs Training Step
*(Only shown if checkpoints contain `global_step`)*

In [ ]:
df_with_step = df.dropna(subset=["global_step"]).sort_values("global_step")

if len(df_with_step) >= 2:
    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle("Metrics vs Training Step", fontsize=13, fontweight="bold")
    axes = axes.flatten()

    for i, (col, title, direction, _) in enumerate(metric_cfg):
        ax = axes[i]
        ax.plot(df_with_step["global_step"], df_with_step[col],
                marker="o", linewidth=2, markersize=6, color=f"C{i}")
        ax.set_xlabel("Global Step")
        ax.set_ylabel(title)
        ax.set_title(f"{title}\n{direction}", fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines[["top","right"]].set_visible(False)
        # Annotate best point
        best_idx = df_with_step[col].idxmax() if _ else df_with_step[col].idxmin()
        brow = df_with_step.loc[best_idx]
        ax.annotate(f"best\n{brow[col]:.4f}",
                    xy=(brow["global_step"], brow[col]),
                    xytext=(10, 10), textcoords="offset points",
                    fontsize=7, color="green",
                    arrowprops=dict(arrowstyle="->", color="green", lw=1))

    # Composite score in last panel
    if "score" in df_with_step.columns:
        ax = axes[5]
        ax.plot(df_with_step["global_step"], df_with_step["score"],
                marker="s", linewidth=2, markersize=7, color="purple")
        ax.set_xlabel("Global Step")
        ax.set_ylabel("Composite Score")
        ax.set_title("Composite Score\n(↑ higher is better)", fontsize=9)
        ax.grid(True, alpha=0.3)
        ax.spines[["top","right"]].set_visible(False)
    else:
        axes[5].set_visible(False)

    plt.tight_layout()
    plt.savefig("/tmp/checkpoint_step_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: /tmp/checkpoint_step_curves.png")
else:
    print("⚠️  Fewer than 2 checkpoints have global_step info — skipping step curve plot.")


## 11 · Per-Clip Distribution (Box Plots)

Shows variance across test clips — stable checkpoints have narrow boxes.

In [ ]:
for col, title, direction, higher_better in metric_cfg:
    raw_per_ckpt = [r["_raw"].get(col, []) for r in results]
    labels = [r["name"][:20] for r in results]

    if all(len(v) == 0 for v in raw_per_ckpt):
        continue

    fig, ax = plt.subplots(figsize=(max(8, 2*len(results)), 4))
    bp = ax.boxplot(raw_per_ckpt, labels=labels, patch_artist=True,
                    medianprops=dict(color="black", linewidth=2))

    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(results)))
    for patch, color in zip(bp["boxes"], colors if higher_better else colors[::-1]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    ax.set_title(f"{title} — per-clip distribution  ({direction})", fontsize=10)
    ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=8)
    ax.grid(True, axis="y", alpha=0.3)
    ax.spines[["top","right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(f"/tmp/boxplot_{col}.png", dpi=120, bbox_inches="tight")
    plt.show()


## 12 · PCA Feature Visualisation

Projects upsampled features to RGB using PCA for qualitative comparison.

In [ ]:
def feats_to_pca_rgb(feats: Tensor, pca: PCA = None):
    """
    feats: (H, W, C) single frame features
    Returns (H, W, 3) uint8 RGB image and fitted PCA object.
    """
    H, W, C = feats.shape
    flat = feats.reshape(-1, C).cpu().numpy().astype(np.float32)
    if pca is None:
        pca = PCA(n_components=3, random_state=0)  # ↓ 3 components = RGB
        pca.fit(flat)
    rgb = pca.transform(flat)                       # (H*W, 3)
    # Normalise to [0, 255]
    rgb -= rgb.min(axis=0)
    mx  = rgb.max(axis=0)
    mx[mx == 0] = 1
    rgb = (rgb / mx * 255).astype(np.uint8)
    return rgb.reshape(H, W, 3), pca


# ── Pick top-K best and bottom-1 checkpoints for visualisation ────────────────
indices_to_vis = list(range(min(TOP_K_CKPTS_PCA, len(results))))
if len(results) > TOP_K_CKPTS_PCA:
    indices_to_vis.append(len(results) - 1)    # worst checkpoint

n_vis       = len(indices_to_vis)
clip_to_vis = 0                                # use first test clip for visualisation
frames_vis  = test_clips[clip_to_vis].to(device)  # (1, T, H, W, 3)

fig, axes = plt.subplots(NUM_FRAMES, n_vis + 1,
                          figsize=(3*(n_vis+1), 2.5*NUM_FRAMES))
fig.suptitle(f"PCA Feature Maps — Clip {clip_to_vis}  (top {TOP_K_CKPTS_PCA} + worst)",
             fontsize=11, fontweight="bold")

# Column 0: raw RGB frames
for t in range(NUM_FRAMES):
    frame_np = (frames_vis[0, t].cpu().numpy() * 255).astype(np.uint8)
    axes[t, 0].imshow(frame_np)
    axes[t, 0].set_title(f"Frame {t}" if t == 0 else "", fontsize=8)
    axes[t, 0].axis("off")
axes[0, 0].set_title("Input RGB", fontsize=9, fontweight="bold")

for col_idx, r_idx in enumerate(indices_to_vis):
    r     = results[r_idx]
    label = ("🏆 BEST" if col_idx == 0 else
             ("💀 WORST" if col_idx == n_vis - 1 and len(results) > TOP_K_CKPTS_PCA
              else f"#{col_idx+1}"))

    try:
        model_vis, _ = load_anyup3d(r["path"])
        pred_vis     = run_model(model_vis, frames_vis)  # (1, T, H', W', C)
        del model_vis
        torch.cuda.empty_cache()

        # Fit PCA on frame 0, apply to all frames
        pca_fitted = None
        for t in range(min(NUM_FRAMES, pred_vis.shape[1])):
            feat_t        = pred_vis[0, t]          # (H', W', C)
            pca_rgb, pca_fitted = feats_to_pca_rgb(feat_t, pca=pca_fitted)
            axes[t, col_idx+1].imshow(pca_rgb)
            axes[t, col_idx+1].axis("off")

        axes[0, col_idx+1].set_title(
            f"{label}\n{r['name'][:18]}\n(step={r['global_step']})",
            fontsize=7, fontweight="bold")
    except Exception as e:
        axes[0, col_idx+1].set_title(f"ERROR\n{e}", fontsize=7, color="red")

plt.tight_layout()
plt.savefig("/tmp/checkpoint_pca_features.png", dpi=130, bbox_inches="tight")
plt.show()
print("Saved: /tmp/checkpoint_pca_features.png")


## 13 · Logged Training Loss Curves
*(Only shown if loss history was saved inside the checkpoint dict)*

In [ ]:
ckpts_with_history = [r for r in results if r["_train_losses"]]

if ckpts_with_history:
    # Discover all logged keys across checkpoints
    all_loss_keys = set()
    for r in ckpts_with_history:
        all_loss_keys.update(r["_train_losses"].keys())
    all_loss_keys = sorted(all_loss_keys)

    fig, axes = plt.subplots(1, len(all_loss_keys),
                              figsize=(6*len(all_loss_keys), 4))
    if len(all_loss_keys) == 1:
        axes = [axes]
    fig.suptitle("Training Loss History (from checkpoint metadata)", fontsize=12)

    for ax, key in zip(axes, all_loss_keys):
        for r in ckpts_with_history:
            if key in r["_train_losses"]:
                vals = r["_train_losses"][key]
                ax.plot(vals, label=r["name"][:20], linewidth=1.5)
        ax.set_title(key, fontsize=10)
        ax.set_xlabel("Step")
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(True, alpha=0.3)
        ax.spines[["top","right"]].set_visible(False)

    plt.tight_layout()
    plt.savefig("/tmp/checkpoint_loss_history.png", dpi=130, bbox_inches="tight")
    plt.show()
else:
    print("No training loss history found in any checkpoint dict.\n"
          "To enable this, save loss_history in your training checkpoint dict:\n"
          "  torch.save({'anyup': model.state_dict(), 'global_step': step,\n"
          "              'loss_history': {'total': [...], 'recon': [...]}}, path)")


## 14 · Final Recommendation

In [ ]:
if len(df) == 0:
    print("No results to display — check errors above.")
else:
    print("=" * 65)
    print("  CHECKPOINT RANKING SUMMARY")
    print("=" * 65)

    for rank, (_, row) in enumerate(df.iterrows(), 1):
        medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, f"#{rank}")
        score_str = f"  score={row.score:.3f}" if "score" in row else ""
        step_str  = f"  step={int(row.global_step)}" if pd.notna(row.global_step) else ""
        print(f"  {medal}  {row['name'][:45]:45s}{score_str}{step_str}")

    best = df.iloc[0]
    print("\n" + "=" * 65)
    print(f"  RECOMMENDATION: load  '{best['path']}'")
    print("=" * 65)
    print(f"\n  cos_sim            = {best['cos_sim']:.5f}  (↑ how close to teacher features)")
    print(f"  cos_mse            = {best['cos_mse']:.5f}  (↓ reconstruction quality)")
    print(f"  temporal_var       = {best['temporal_var']:.5f}  (↓ temporal stability)")
    print(f"  cosine_drift       = {best['cosine_drift']:.5f}  (↓ frame-to-frame coherence)")
    print(f"  input_consistency  = {best['input_consistency']:.5f}  (↓ preserves input features)")

    print("\n  To load this checkpoint:")
    print(f"  ckpt = torch.load('{best['path']}', map_location='cpu')")
    print("  model.load_state_dict(ckpt['anyup'], strict=False)")
